# Hosted Feature Service Monitor (ArcGIS Online)

This notebook helps an ArcGIS Online administrator:

1. Find the 10 most recently created hosted feature services in the organization.
2. Build one alert email to an administrator group if any of those items are publicly shared.

Design choices used here:
- Runtime/auth: ArcGIS Online notebook with `GIS("home")`
- Hosted filter: only hosted feature services (not files, tiles, imagery, or other item types)
- Public trigger: only items shared to everyone
- One email per run, test mode first
- Time shown in UTC for consistency

In [ ]:
from datetime import datetime, timezone

import pandas as pd
from arcgis.gis import GIS

# ---------- Config ----------
TOP_N = 10
SEARCH_POOL_SIZE = 200

# Set ADMIN_GROUP_ID if you know it. This is the most reliable option.
ADMIN_GROUP_ID = ""

# Optional fallback if ID is not set. Should match group title exactly.
ADMIN_GROUP_TITLE = ""

# Test mode prints the exact message instead of sending it.
TEST_MODE = True

# Only set this True when you are ready to send.
SEND_EMAIL = False

# If True, hosted views are included.
INCLUDE_HOSTED_VIEWS = True

In [ ]:
def to_utc_string(ms_since_epoch: int) -> str:
    if not ms_since_epoch:
        return ""
    return datetime.fromtimestamp(ms_since_epoch / 1000, tz=timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")


def sharing_label(access_value: str) -> str:
    mapping = {
        "public": "public",
        "everyone": "public",
        "org": "organization",
        "private": "private",
        "shared": "group",
    }
    return mapping.get((access_value or "").lower(), access_value or "unknown")


def is_hosted_feature_service(item) -> bool:
    if (item.type or "").lower() != "feature service":
        return False
    keywords = {str(k).strip().lower() for k in (item.typeKeywords or [])}
    if "hosted service" not in keywords:
        return False
    if not INCLUDE_HOSTED_VIEWS and "view service" in keywords:
        return False
    return True

In [ ]:
gis = GIS("home")
print(f"Connected to: {gis.properties.portalHostname}")
print(f"Signed in as: {gis.users.me.username}")

search_results = gis.content.search(
    query="",
    item_type="Feature Service",
    sort_field="created",
    sort_order="desc",
    max_items=SEARCH_POOL_SIZE,
)

hosted_items = [item for item in search_results if is_hosted_feature_service(item)]
latest_hosted_items = hosted_items[:TOP_N]

records = []
for item in latest_hosted_items:
    records.append(
        {
            "title": item.title,
            "owner": item.owner,
            "created_utc": to_utc_string(item.created),
            "sharing_level": sharing_label(item.access),
            "num_views": item.numViews if item.numViews is not None else 0,
            "item_id": item.id,
        }
    )

latest_df = pd.DataFrame(records)

if latest_df.empty:
    print("No hosted feature services were found in the current search window.")
else:
    display(latest_df)
    print(f"Displayed {len(latest_df)} hosted feature services (most recent first).")

## Email Alert Cell

This cell sends or previews exactly one consolidated email message when public items are found among the top 10 list.

Safe defaults:
- `TEST_MODE = True`
- `SEND_EMAIL = False`

To send for real, set `TEST_MODE = False` and `SEND_EMAIL = True` in the config cell above.

In [ ]:
if "latest_df" not in globals() or latest_df.empty:
    print("No candidate hosted feature services to evaluate for alerting.")
else:
    public_df = latest_df[latest_df["sharing_level"].str.lower() == "public"].copy()

    if public_df.empty:
        print("No public items detected in the top hosted feature services. No email needed.")
    else:
        if not ADMIN_GROUP_ID and not ADMIN_GROUP_TITLE:
            print("Set ADMIN_GROUP_ID (preferred) or ADMIN_GROUP_TITLE before sending alerts.")
        else:
            group = None
            if ADMIN_GROUP_ID:
                group = gis.groups.get(ADMIN_GROUP_ID)
            elif ADMIN_GROUP_TITLE:
                matches = gis.groups.search(query=f'title:"{ADMIN_GROUP_TITLE}"', max_groups=5)
                exact = [g for g in matches if (g.title or "") == ADMIN_GROUP_TITLE]
                group = exact[0] if exact else None

            if group is None:
                print("Administrator group could not be resolved. Check group ID/title.")
            else:
                members = group.get_members() or {}
                recipients = sorted(
                    set((members.get("users") or []) + (members.get("admins") or []) + ([members.get("owner")] if members.get("owner") else []))
                )

                if not recipients:
                    print("No recipients found in the administrator group.")
                else:
                    subject = f"ArcGIS Online alert: {len(public_df)} public hosted feature service(s) detected"

                    lines = [
                        "The following hosted feature services are publicly shared:",
                        "",
                    ]
                    for _, row in public_df.iterrows():
                        lines.append(
                            f"- {row['title']} (owner: {row['owner']}, created: {row['created_utc']}, views: {row['num_views']}, id: {row['item_id']})"
                        )

                    lines.extend(
                        [
                            "",
                            "This message was generated by the Hosted Feature Service Monitor notebook.",
                        ]
                    )
                    message = "\n".join(lines)

                    print(f"Recipients ({len(recipients)}): {', '.join(recipients)}")

                    if TEST_MODE:
                        print("\n[TEST MODE] Email preview only. Nothing sent.\n")
                        print("Subject:")
                        print(subject)
                        print("\nBody:")
                        print(message)
                        if SEND_EMAIL:
                            print("\nWARNING: SEND_EMAIL is True, but TEST_MODE prevents sending. Set TEST_MODE = False to send.")
                    elif SEND_EMAIL:
                        try:
                            result = group.notify(users=recipients, subject=subject, message=message)
                            print("Notification send result:")
                            print(result)
                        except Exception as ex:
                            print("Sending failed. Check group permissions and notify support in your org.")
                            print(f"Error: {ex}")
                    else:
                        print("SEND_EMAIL is False. No email sent.")